# F1 2025-2026: Tire Strategy and Weather

Uses the `tire_stints.csv`, `weather.csv`, and `practice_results.csv` tables added for the 2025/2026 seasons (sourced from official F1 live timing via FastF1). These aren't available for 1950-2024, so this notebook is scoped to the two most recent seasons.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tire_stints.csv" in files:
        DATA_DIR = root
        break
print("Using data dir:", DATA_DIR)

races = pd.read_csv(f"{DATA_DIR}/races.csv")
drivers = pd.read_csv(f"{DATA_DIR}/drivers.csv")
stints = pd.read_csv(f"{DATA_DIR}/tire_stints.csv")
weather = pd.read_csv(f"{DATA_DIR}/weather.csv")

print(f"Tire stint rows: {len(stints)}   Weather rows: {len(weather)}   Races with tire data: {stints.raceId.nunique()}")

## Tire strategy chart for a race

Classic F1 strategy visualization: each driver's stints, colored by compound. Change `RACE_NAME` / `RACE_YEAR` to look at any 2025/2026 race.

In [ ]:
RACE_NAME = "Hungarian Grand Prix"
RACE_YEAR = 2026

race_id = races[(races.name == RACE_NAME) & (races.year == RACE_YEAR)].raceId.iloc[0]
race_stints = stints[stints.raceId == race_id].merge(drivers[["driverId", "code", "surname"]], on="driverId")

compound_colors = {
    "SOFT": "#DA291C", "MEDIUM": "#FFD12E", "HARD": "#F0F0F0",
    "INTERMEDIATE": "#43B02A", "WET": "#0067AD",
}

driver_order = race_stints.groupby("driverId").endLap.max().sort_values(ascending=False).index
driver_labels = drivers.set_index("driverId").loc[driver_order, "surname"]

fig, ax = plt.subplots(figsize=(12, 8))
for y, driver_id in enumerate(driver_order):
    driver_stints = race_stints[race_stints.driverId == driver_id].sort_values("stint")
    for _, row in driver_stints.iterrows():
        color = compound_colors.get(row.compound, "#888888")
        ax.barh(y, row.lapsOnTire, left=row.startLap - 1, color=color, edgecolor="black", height=0.7)

ax.set_yticks(range(len(driver_order)))
ax.set_yticklabels(driver_labels)
ax.invert_yaxis()
ax.set_xlabel("Lap")
ax.set_title(f"{RACE_NAME} {RACE_YEAR}: tire strategy")

handles = [plt.Rectangle((0,0),1,1, color=c) for c in compound_colors.values()]
ax.legend(handles, compound_colors.keys(), loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=5)
plt.tight_layout()
plt.show()

## Weather across the 2025-2026 season

In [ ]:
w = weather.merge(races[["raceId", "name", "year", "date"]], on="raceId").sort_values("date")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(w.date, w.airTempAvg, label="Air temp (avg)", marker="o")
ax.plot(w.date, w.trackTempAvg, label="Track temp (avg)", marker="o")
for _, row in w[w.rainfall].iterrows():
    ax.axvline(row.date, color="blue", alpha=0.2)
ax.set_ylabel("\u00b0C")
ax.set_title("Air & track temperature by race (blue lines = rainfall recorded)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Rain races:")
print(w[w.rainfall][["name", "year"]].to_string(index=False))

## Most-used tire compound by race

In [ ]:
compound_counts = stints.groupby(["raceId", "compound"]).size().unstack(fill_value=0)
compound_counts = compound_counts.merge(races[["raceId", "name", "year"]], on="raceId").set_index("name")
compound_cols = [c for c in ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"] if c in compound_counts.columns]
compound_counts[compound_cols].tail(11).plot(kind="bar", stacked=True, figsize=(14, 5),
                                               color=[compound_colors.get(c, "#888") for c in compound_cols])
plt.title("Tire compound usage, most recent 11 races")
plt.ylabel("Stints")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()